# Pattern #4: Multi-Agent Systems - Parallel Specialists

**From-Scratch Implementation**

## Overview

Multi-agent systems use specialized agents working in parallel:

```
Query → Router (intent + patient id) → ┌─ PolicyAgent (RAG)     ─┐
                                       ├─ LogisticsAgent (Web)  ─┤
                                       └─ BookingLookup (tool)  ─┘
                                                 ↓
                                              Judge
                                                 ↓
                                            Final Answer
```

**Router:** Identifies intent (policy / logistics / booking) and extracts patient name/phone/email when the user asks about "my appointment".

**Agents & tool:**
- **PolicyAgent**: Internal RAG for procedures, guidelines, clinical info, what to bring
- **LogisticsAgent**: Web search for hours, contacts, operational info
- **BookingLookup**: Tool that looks up patient bookings by name/phone/email from `data/bookings.json`
- **Judge**: Merges results from the selected agents/tool

**Benefits**: Specialization + Parallel Execution = Better Performance

In [ ]:
import sys
from pathlib import Path
sys.path.append('..')

from utils import create_llm_provider, get_config
from rag_internal import MedicalKnowledgeRetriever, get_retriever
from tools import get_web_search_tool, get_booking_tool
import concurrent.futures
import json
import re

# Resolve path to data/medical_guides (works whether cwd is project root or scratch_demos)
_project_root = Path.cwd() if (Path.cwd() / "data" / "medical_guides").exists() else Path.cwd().parent
DOCS_DIR = _project_root / "data" / "medical_guides"

# Initialize
config = get_config()
llm = create_llm_provider()
retriever = MedicalKnowledgeRetriever(docs_directory=str(DOCS_DIR))
web_search = get_web_search_tool()
booking_tool = get_booking_tool()

# Ensure documents indexed
if not retriever._is_indexed:
    print("Indexing documents...")
    stats = retriever.index_documents()
    status = stats.get("status", "unknown")
    if status == "indexed":
        print(f"Indexed {stats.get('chunk_count', 0)} chunks from {stats.get('document_count', 0)} documents")
    elif status in ("no_documents", "no_content"):
        print("No documents in data directory. Add .txt/.pdf/.docx to data/medical_guides/ and re-run.")
    else:
        print(f"Status: {status}, documents: {stats.get('document_count', 0)}")
else:
    print(f"Documents indexed: {retriever.get_stats()['total_chunks']} chunks")

print(f"\nUsing model: {config.get('model')}")
print("Multi-agent system: Router + PolicyAgent + LogisticsAgent + Booking tool + Judge")


### Validation (config from `config/`, Router + agents + booking tool)

In [ ]:
assert config.get("model"), "config.get('model') should be set"
assert llm is not None, "create_llm_provider() should return a provider"
assert retriever._is_indexed, "Retriever should be indexed"
assert callable(web_search.search), "web_search tool ready"
assert hasattr(booking_tool, "lookup"), "booking_tool ready"
print(" Setup valid: config, LLM, retriever, web_search, and booking_tool ready.")

## Multi-Agent Implementation


In [ ]:
from typing import Dict, Any
import time

try:
    from IPython.display import display, Markdown
except ImportError:
    display = print
    def Markdown(s): return s

class PolicyAgent:
    """Specialist for medical policies and procedures using internal RAG."""
    
    def __init__(self, llm, retriever):
        self.llm = llm
        self.retriever = retriever
        self.name = "PolicyAgent"
    
    def process(self, query: str) -> Dict[str, Any]:
        """Process query using internal RAG."""
        context = self.retriever.get_context(query, max_k=3)
        
        system_prompt = """
You are a medical policy specialist.
Use internal knowledge to answer questions about procedures, guidelines, and clinical protocols.
Be specific and cite sources.
""".strip()
        
        prompt = f"""
Query: {query}

Internal Knowledge:
{context}

Provide a focused answer about medical policies and procedures.
""".strip()
        
        response = self.llm.generate(prompt=prompt, system_prompt=system_prompt)
        
        return {
            "agent": self.name,
            "response": response["response"],
            "tokens": response["total_tokens"],
            "latency_ms": response["latency_ms"],
            "source_type": "internal_rag"
        }


class LogisticsAgent:
    """Specialist for operational logistics using web search."""
    
    def __init__(self, llm, web_search):
        self.llm = llm
        self.web_search = web_search
        self.name = "LogisticsAgent"
    
    def process(self, query: str) -> Dict[str, Any]:
        """Process query using web search."""
        search_result = self.web_search.search(query, max_results=3)
        search_text = self.web_search.format_results(search_result)
        
        system_prompt = """
You are a healthcare logistics specialist.
Use web search results to answer questions about hours, contacts, locations, and current operations.
Include source URLs.
""".strip()
        
        prompt = f"""
Query: {query}

Web Search Results:
{search_text}

Provide a focused answer about operational logistics.
""".strip()
        
        response = self.llm.generate(prompt=prompt, system_prompt=system_prompt)
        
        return {
            "agent": self.name,
            "response": response["response"],
            "tokens": response["total_tokens"],
            "latency_ms": response["latency_ms"],
            "source_type": "web_search"
        }


class Router:
    """Identifies intent and extracts patient identifier for booking lookups."""
    
    def __init__(self, llm):
        self.llm = llm
    
    def route(self, query: str) -> Dict[str, Any]:
        """Classify intent (policy, logistics, booking) and extract patient identifier if booking-related."""
        system_prompt = """You are a router for a healthcare multi-agent system .
Classify the user's intent and extract any patient identifier they mention.

Intent:
- policy: Questions about medical procedures, guidelines, clinical protocols, what to bring for a condition, preparation (use internal knowledge/RAG).
- logistics: Questions about hospital hours, locations, contacts, operational info (use web search).
- booking: Questions about the user's own appointment(s)—when, where, with whom, status, notes (use booking lookup by patient name/phone/email).

Extract patient identifier only when booking is true: patient_name (e.g. "John Silva"), patient_phone, or patient_email from the message. Use null for missing.

Respond with ONLY a single JSON object, no markdown, no explanation. Keys: policy (boolean), logistics (boolean), booking (boolean), patient_name (string or null), patient_phone (string or null), patient_email (string or null)."""
        prompt = f"User query: {query}"
        response = self.llm.generate(prompt=prompt, system_prompt=system_prompt)
        text = (response.get("response") or "").strip()
        # Strip markdown code block if present
        if "```" in text:
            text = re.sub(r"^```(?:json)?\s*", "", text)
            text = re.sub(r"\s*```\s*$", "", text)
        try:
            out = json.loads(text)
        except json.JSONDecodeError:
            out = {"policy": True, "logistics": True, "booking": False, "patient_name": None, "patient_phone": None, "patient_email": None}
        for key in ("policy", "logistics", "booking"):
            out[key] = bool(out.get(key))
        for key in ("patient_name", "patient_phone", "patient_email"):
            v = out.get(key)
            out[key] = (v.strip() if isinstance(v, str) and v.strip() else None) or None
        return out


class Judge:
    """Merges results from multiple agents."""
    
    def __init__(self, llm):
        self.llm = llm
    
    def merge(self, query: str, agent_results: list) -> Dict[str, Any]:
        """Merge agent responses into final answer."""
        merged_info = ""
        for result in agent_results:
            merged_info += f"\n\n{result['agent']} ({result['source_type']}):\n{result['response']}"
        
        system_prompt = """
You are a judge synthesizing information from specialist agents.
Create a comprehensive, well-organized final answer.
Give priority to internal RAG for medical info, web search for operational info.
Always end with: "This is educational information; verify with the hospital / consult your clinician."
""".strip()
        
        prompt = f"""
User Query: {query}

Specialist Agent Responses:
{merged_info}

Synthesize these into a clear, comprehensive final answer.
""".strip()
        
        response = self.llm.generate(prompt=prompt, system_prompt=system_prompt)
        
        return {
            "final_answer": response["response"],
            "tokens": response["total_tokens"],
            "latency_ms": response["latency_ms"]
        }


class MultiAgentSystem:
    """Orchestrates Router, specialist agents, and booking tool."""
    
    def __init__(self, llm, retriever, web_search, booking_tool):
        self.router = Router(llm)
        self.policy_agent = PolicyAgent(llm, retriever)
        self.logistics_agent = LogisticsAgent(llm, web_search)
        self.booking_tool = booking_tool
        self.judge = Judge(llm)
    
    def run(self, query: str, parallel: bool = True, verbose: bool = True) -> Dict[str, Any]:
        """Run multi-agent system: Router → selected agents + booking (if needed) → Judge."""
        start_time = time.time()
        
        if verbose:
            display(Markdown(f"**Query:** {query}\n"))
        
        # 1. Router: intent + patient identifier
        route = self.router.route(query)
        need_policy = route.get("policy", True)
        need_logistics = route.get("logistics", True)
        need_booking = route.get("booking", False)
        if verbose:
            display(Markdown(f"**Router:** `policy={need_policy}`, `logistics={need_logistics}`, `booking={need_booking}`  \nPatient: `{route.get('patient_name') or route.get('patient_phone') or route.get('patient_email') or '—'}`\n"))
        
        agent_results = []
        
        # 2. Booking lookup (when intent is booking and we have an identifier)
        if need_booking:
            bookings = self.booking_tool.lookup(
                patient_name=route.get("patient_name"),
                patient_phone=route.get("patient_phone"),
                patient_email=route.get("patient_email"),
            )
            booking_text = self.booking_tool.format_bookings_for_agent(bookings)
            if not (route.get("patient_name") or route.get("patient_phone") or route.get("patient_email")):
                booking_text = "Booking intent detected but no patient name, phone, or email was found in the message. Please ask the user to identify themselves (e.g. 'I'm John Silva') to look up their appointment."
            agent_results.append({
                "agent": "BookingLookup",
                "response": booking_text,
                "tokens": 0,
                "latency_ms": 0,
                "source_type": "booking_lookup",
            })
            if verbose:
                display(Markdown(f"### BookingLookup (tool)\n\n{booking_text}\n"))
        
        # 3. Run Policy and/or Logistics (parallel when both needed)
        policy_result = None
        logistics_result = None
        if need_policy or need_logistics:
            if parallel and need_policy and need_logistics:
                if verbose:
                    display(Markdown("*Executing Policy + Logistics in parallel...*\n"))
                with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
                    future_policy = executor.submit(self.policy_agent.process, query) if need_policy else None
                    future_logistics = executor.submit(self.logistics_agent.process, query) if need_logistics else None
                    policy_result = future_policy.result() if future_policy else None
                    logistics_result = future_logistics.result() if future_logistics else None
            else:
                if verbose and (need_policy or need_logistics):
                    display(Markdown("*Executing selected agents...*\n"))
                if need_policy:
                    policy_result = self.policy_agent.process(query)
                if need_logistics:
                    logistics_result = self.logistics_agent.process(query)
            
            if policy_result:
                agent_results.append(policy_result)
                if verbose:
                    display(Markdown(f"### PolicyAgent (RAG) — {policy_result['latency_ms']}ms\n\n{policy_result['response']}\n"))
            if logistics_result:
                agent_results.append(logistics_result)
                if verbose:
                    display(Markdown(f"### LogisticsAgent (Web) — {logistics_result['latency_ms']}ms\n\n{logistics_result['response']}\n"))
        
        # If router said no policy and no logistics and no booking, run both agents as fallback
        if not agent_results:
            if verbose:
                display(Markdown("*No intent matched; running Policy + Logistics as fallback...*\n"))
            with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
                future_policy = executor.submit(self.policy_agent.process, query)
                future_logistics = executor.submit(self.logistics_agent.process, query)
                policy_result = future_policy.result()
                logistics_result = future_logistics.result()
            agent_results = [policy_result, logistics_result]
            if verbose:
                display(Markdown(f"*PolicyAgent* ({policy_result['latency_ms']}ms) · *LogisticsAgent* ({logistics_result['latency_ms']}ms)\n"))
        
        if verbose:
            display(Markdown("---\n### JUDGE: Merging Results\n"))
        
        judge_result = self.judge.merge(query, agent_results)
        
        if verbose:
            display(Markdown("#### Final answer\n"))
            display(Markdown(judge_result["final_answer"]))
        
        end_time = time.time()
        total_time_ms = int((end_time - start_time) * 1000)
        total_tokens = judge_result["tokens"] + sum(r.get("tokens", 0) for r in agent_results)
        
        if verbose:
            display(Markdown(f"---\n**Summary:** Router → policy={need_policy}, logistics={need_logistics}, booking={need_booking}  \nTotal tokens: {total_tokens} · Time: {total_time_ms}ms"))
        
        return {
            "query": query,
            "route": route,
            "policy_result": policy_result,
            "logistics_result": logistics_result,
            "final_answer": judge_result["final_answer"],
            "total_tokens": total_tokens,
            "total_time_ms": total_time_ms,
            "parallel": parallel,
        }

# Create system (with booking tool)
system = MultiAgentSystem(llm, retriever, web_search, booking_tool)
print("Multi-agent system initialized (Router + Policy + Logistics + Booking + Judge) ✓")


## Example: Complex Multi-Domain Query (Policy + Logistics + Booking)

One query that triggers **all three**: treatment guidelines (Policy/RAG), hospital info (Logistics/web), and the user’s own appointment (Booking lookup). The Router detects all intents and the Judge merges everything into one answer.


In [ ]:
# Triggers: policy (guidelines) + logistics (hospital/dermatology) + booking (extract "John Silva", lookup appointment)
query = "I'm John Silva. What are the treatment guidelines for psoriasis, which hospital in Colombo has the best dermatology department open today, and when is my appointment?"
result = system.run(query, parallel=True, verbose=True)


## Example Queries: Router Intent + Booking

Each example runs **separately** so you can see exactly what the Router triggers. Run the cells one at a time.

In [ ]:
# --- 1. Booking-only ---
# Trigger: Router should set booking=True, extract patient_name="John Silva", run only BookingLookup (no Policy/Logistics).
query = "I'm John Silva – when is my appointment?"
result = system.run(query, parallel=True, verbose=True)

In [ ]:
# --- 2. Mixed (booking + policy) ---
# Trigger: Router sets booking=True (extract "Mary Fernando") and policy=True ("what should I bring").
# Expect: BookingLookup + PolicyAgent (RAG); Judge merges appointment details + preparation advice.
query = "I'm Mary Fernando. When is my appointment and what should I bring?"
result = system.run(query, parallel=True, verbose=True)

In [ ]:
# --- 3. Policy-only ---
# Trigger: Router sets policy=True, logistics=False, booking=False.
# Expect: Only PolicyAgent (RAG) runs; internal guidelines for psoriasis.
query = "What are the treatment guidelines for psoriasis?"
result = system.run(query, parallel=True, verbose=True)

In [ ]:
# --- 4. Logistics-only ---
# Trigger: Router sets logistics=True, policy=False, booking=False.
# Expect: Only LogisticsAgent (web search) runs; hospital/dermatology info.
query = "Which hospital in Colombo has the best dermatology department open today?"
result = system.run(query, parallel=True, verbose=True)

## Pattern Summary

Multi-agent systems leverage **specialization** and **parallel execution**:

**Architecture:**
```
Query
  ↓
Router (implicit via parallel execution)
  ├─→ PolicyAgent (RAG) ─┐
  └─→ LogisticsAgent (Web)─┤
                           ↓
                         Judge
                           ↓
                      Final Answer
```

**Key Features:**
- **Specialization**: Each agent has specific expertise
- **Parallel Execution**: Agents work simultaneously
- **Intelligent Merging**: Judge synthesizes results appropriately

**Benefits:**
- **Speed**: Parallel > Sequential (lower latency)
- **Quality**: Specialists > Generalists (better focus)
- **Scalability**: Easy to add more specialist agents
- **Clear Separation**: Internal (RAG) vs External (Web) expertise

**When to use:**
- Complex queries spanning multiple domains
- Need for both policy and operational info
- Speed is important (parallel execution helps)
- Clear specialization boundaries exist

**Trade-offs:**
- More complex architecture
- Need orchestration (router + judge)
- Higher token cost (multiple agents + judge)
- Potential for conflicting information (judge must resolve)

**Comparison with ReAct:**
- ReAct: Sequential, adaptive, visible reasoning
- Multi-Agent: Parallel, specialized, distributed processing

Multi-agent systems scale well to complex real-world applications!
